# Protein Sequence Classification with ProtT5 and LoRA

In [ ]:
!pip install torchao --upgrade -q
!pip install transformers datasets sentencepiece peft scikit-learn -q

## Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import random
import re
import time
import gc
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import f1_score, accuracy_score
from datasets import load_dataset
from transformers import T5EncoderModel, T5Tokenizer
from peft import get_peft_model, LoraConfig
from collections import Counter

print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))
print("Memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

NUM_CLASSES = 20
HIDDEN_SIZE = 1024
MAX_LENGTH  = 256

GPU available: True
GPU name: Tesla T4
Memory: 15.6 GB


In [ ]:
print("Loading dataset...")
dataset    = load_dataset("lightonai/SwissProt-EC-leaf")
train_data = dataset["train"]
val_data   = dataset["dev"]
test_data  = dataset["test"]

# Find top 20 most common EC classes in training set
label_counts = Counter()
for example in train_data:
    label_counts[example["labels"][0]] += 1

top20_labels = set([label for label, _ in label_counts.most_common(20)])
print("Top 20 label indices:", sorted(top20_labels))

# Filter to top 20 only
def filter_top20(data, top20_labels):
    return [ex for ex in data if ex["labels"][0] in top20_labels]

train_filtered = filter_top20(train_data, top20_labels)
val_filtered   = filter_top20(val_data,   top20_labels)
test_filtered  = filter_top20(test_data,  top20_labels)

print(f"Train: {len(train_data)} → {len(train_filtered)}")
print(f"Val:   {len(val_data)}   → {len(val_filtered)}")
print(f"Test:  {len(test_data)}  → {len(test_filtered)}")

# Remap labels to 0-19
label_map = {old: new for new, old in enumerate(sorted(top20_labels))}

# Sample to match Shruti's setup exactly
random.seed(42)
train_sample = random.sample(train_filtered, 5000)
val_sample   = random.sample(val_filtered,   min(1000, len(val_filtered)))
test_sample  = random.sample(test_filtered,  min(1000, len(test_filtered)))

print(f"\nFinal sizes:")
print(f"Train: {len(train_sample)}")
print(f"Val:   {len(val_sample)}")
print(f"Test:  {len(test_sample)}")

Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Top 20 label indices: [309, 492, 662, 871, 1613, 2042, 2084, 2302, 2743, 3174, 3315, 3695, 3746, 3901, 3971, 3982, 4088, 4514, 4518, 4521]
Train: 178302 → 22099
Val:   23010   → 2788
Test:  22183  → 2734

Final sizes:
Train: 5000
Val:   1000
Test:  1000


## Data Loading and Preprocessing

In [ ]:
print("Loading tokenizer...")
tokenizer = T5Tokenizer.from_pretrained(
    "Rostlab/prot_t5_xl_uniref50",
    do_lower_case=False
)

print("Loading ProtT5 (3-5 mins)...")
model = T5EncoderModel.from_pretrained(
    "Rostlab/prot_t5_xl_uniref50",
    torch_dtype=torch.float16
)
model = model.to("cuda")
model.eval()

print("Parameters:", round(sum(p.numel() for p in model.parameters())/1e6), "M")
print("Memory free:", round(torch.cuda.mem_get_info()[0]/1e9, 1), "GB")

Loading tokenizer...
Loading ProtT5 (3-5 mins)...


pytorch_model.bin:   0%|          | 0.00/11.3G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/11.3G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
T5EncoderModel LOAD REPORT from: Rostlab/prot_t5_xl_uniref50
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Parameters: 1208 M
Memory free: 12.1 GB


## Model and Tokenizer Loading

In [ ]:
class ProteinDataset(Dataset):
    def __init__(self, data, tokenizer, label_map, max_length=256):
        self.data       = data
        self.tokenizer  = tokenizer
        self.label_map  = label_map
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        example = self.data[idx]
        seq     = " ".join(list(example["seq"]))
        seq     = re.sub(r"[UZOB]", "X", seq)
        label   = self.label_map[example["labels"][0]]
        return seq, label

def collate_fn(batch):
    seqs, labels = zip(*batch)
    encoded = tokenizer(
        list(seqs),
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    )
    return encoded, torch.tensor(labels, dtype=torch.long)

train_dataset = ProteinDataset(train_sample, tokenizer, label_map)
val_dataset   = ProteinDataset(val_sample,   tokenizer, label_map)
test_dataset  = ProteinDataset(test_sample,  tokenizer, label_map)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False, collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")

Train batches: 625
Val batches:   125


## Dataset and DataLoader Preparation

In [ ]:
def compute_embeddings(dataset, batch_size=16):
    all_embeddings = []
    all_labels     = []
    loader = DataLoader(dataset, batch_size=batch_size,
                       shuffle=False, collate_fn=collate_fn)
    start = time.time()

    for i, (encoded, labels) in enumerate(loader):
        encoded = {k: v.to("cuda") for k, v in encoded.items()}
        with torch.no_grad():
            outputs    = model(**encoded)
            embeddings = outputs.last_hidden_state.mean(dim=1).float().cpu()
        all_embeddings.append(embeddings)
        all_labels.append(labels)

        if (i+1) % 20 == 0:
            print(f"  {(i+1)*batch_size}/{len(dataset)} | "
                  f"{(time.time()-start)/60:.1f}min")

    return torch.cat(all_embeddings), torch.cat(all_labels)

from torch.utils.data import TensorDataset

print("Computing train embeddings...")
train_emb, train_labels = compute_embeddings(train_dataset)
print("Computing val embeddings...")
val_emb, val_labels = compute_embeddings(val_dataset)

torch.save((train_emb, train_labels), "train_embeddings_top20.pt")
torch.save((val_emb,   val_labels),   "val_embeddings_top20.pt")

fast_train = DataLoader(TensorDataset(train_emb, train_labels),
                       batch_size=256, shuffle=True)
fast_val   = DataLoader(TensorDataset(val_emb,   val_labels),
                       batch_size=256, shuffle=False)

print(f"Done! Train: {train_emb.shape} | Val: {val_emb.shape}")

Computing train embeddings...
  320/5000 | 0.5min
  640/5000 | 0.9min
  960/5000 | 1.3min
  1280/5000 | 1.8min
  1600/5000 | 2.3min
  1920/5000 | 2.7min
  2240/5000 | 3.1min
  2560/5000 | 3.6min
  2880/5000 | 4.0min
  3200/5000 | 4.5min
  3520/5000 | 4.9min
  3840/5000 | 5.4min
  4160/5000 | 5.8min
  4480/5000 | 6.3min
  4800/5000 | 6.7min
Computing val embeddings...
  320/1000 | 0.4min
  640/1000 | 0.9min
  960/1000 | 1.3min
Done! Train: torch.Size([5000, 1024]) | Val: torch.Size([1000, 1024])


## Embedding Computation

## Training Strategies

### Strategy 1: Linear Probing

In [ ]:
print("="*50)
print("STRATEGY 1: LINEAR PROBING")
print("="*50)

# Freeze everything
for param in model.parameters():
    param.requires_grad = False

classifier = nn.Linear(HIDDEN_SIZE, NUM_CLASSES).to("cuda")
optimizer  = AdamW(classifier.parameters(), lr=1e-3)
criterion  = nn.CrossEntropyLoss()

lp_start = time.time()

NUM_EPOCHS = 10
for epoch in range(NUM_EPOCHS):
    classifier.train()
    total_loss, correct, total = 0, 0, 0

    for emb_batch, label_batch in fast_train:
        emb_batch   = emb_batch.to("cuda")
        label_batch = label_batch.to("cuda")
        logits      = classifier(emb_batch)
        loss        = criterion(logits, label_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct    += (logits.argmax(1) == label_batch).sum().item()
        total      += label_batch.size(0)

    # Validate
    classifier.eval()
    all_preds, all_labels_list = [], []
    with torch.no_grad():
        for emb_batch, label_batch in fast_val:
            logits = classifier(emb_batch.to("cuda"))
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels_list.extend(label_batch.numpy())

    val_acc = accuracy_score(all_labels_list, all_preds)
    val_f1  = f1_score(all_labels_list, all_preds,
                      average="macro", zero_division=0)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Loss: {total_loss/len(fast_train):.4f} | "
          f"Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

lp_time = time.time() - lp_start
torch.save(classifier.state_dict(), "linear_probe_top20.pt")

print(f"\nLINEAR PROBING RESULTS:")
print(f"Val Accuracy: {val_acc:.4f}")
print(f"Val Macro F1: {val_f1:.4f}")
print(f"Runtime:      {lp_time:.2f}s")
lp_acc, lp_f1 = val_acc, val_f1

STRATEGY 1: LINEAR PROBING
Epoch 1/10 | Loss: 2.8427 | Val Acc: 0.5580 | Val F1: 0.3386
Epoch 2/10 | Loss: 2.5448 | Val Acc: 0.5850 | Val F1: 0.3894
Epoch 3/10 | Loss: 2.2874 | Val Acc: 0.6570 | Val F1: 0.5300
Epoch 4/10 | Loss: 2.0645 | Val Acc: 0.7460 | Val F1: 0.7073
Epoch 5/10 | Loss: 1.8657 | Val Acc: 0.8290 | Val F1: 0.8189
Epoch 6/10 | Loss: 1.6885 | Val Acc: 0.8620 | Val F1: 0.8564
Epoch 7/10 | Loss: 1.5301 | Val Acc: 0.8860 | Val F1: 0.8858
Epoch 8/10 | Loss: 1.3881 | Val Acc: 0.8960 | Val F1: 0.9008
Epoch 9/10 | Loss: 1.2608 | Val Acc: 0.9120 | Val F1: 0.9240
Epoch 10/10 | Loss: 1.1494 | Val Acc: 0.9160 | Val F1: 0.9292

LINEAR PROBING RESULTS:
Val Accuracy: 0.9160
Val Macro F1: 0.9292
Runtime:      3.17s


### Strategy 2: LoRA Fine-tuning

## Training Strategies

In [ ]:
print("="*50)
print("STRATEGY 2: LoRA")
print("="*50)

# Clear memory
gc.collect()
torch.cuda.empty_cache()

# Apply LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    target_modules=["q", "v"]
)
lora_model = get_peft_model(model, lora_config)
lora_model.gradient_checkpointing_enable()
lora_model.print_trainable_parameters()

classifier_lora = nn.Linear(HIDDEN_SIZE, NUM_CLASSES).to("cuda")
criterion       = nn.CrossEntropyLoss()
optimizer_lora  = AdamW([
    {"params": lora_model.parameters(),      "lr": 5e-5},
    {"params": classifier_lora.parameters(), "lr": 1e-3}
])

lora_train_loader = DataLoader(train_dataset, batch_size=4,
                               shuffle=True,  collate_fn=collate_fn)
lora_val_loader   = DataLoader(val_dataset,   batch_size=4,
                               shuffle=False, collate_fn=collate_fn)

lora_start = time.time()
NUM_EPOCHS = 3

for epoch in range(NUM_EPOCHS):
    lora_model.train()
    classifier_lora.train()
    total_loss, correct, total = 0, 0, 0

    for batch_idx, (encoded, labels) in enumerate(lora_train_loader):
        labels  = labels.to("cuda")
        encoded = {k: v.to("cuda") for k, v in encoded.items()}

        outputs   = lora_model(**encoded)
        embedding = outputs.last_hidden_state.mean(dim=1).float()
        logits    = classifier_lora(embedding)
        loss      = criterion(logits, labels)

        optimizer_lora.zero_grad()
        loss.backward()
        optimizer_lora.step()

        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)

        if (batch_idx+1) % 100 == 0:
            print(f"  Batch {batch_idx+1}/{len(lora_train_loader)} | "
                  f"Loss: {total_loss/(batch_idx+1):.4f} | "
                  f"Acc: {correct/total*100:.1f}%")

    # Validate
    lora_model.eval()
    classifier_lora.eval()
    all_preds, all_labels_list = [], []
    with torch.no_grad():
        for encoded, labels in lora_val_loader:
            labels  = labels.to("cuda")
            encoded = {k: v.to("cuda") for k, v in encoded.items()}
            outputs   = lora_model(**encoded)
            embedding = outputs.last_hidden_state.mean(dim=1).float()
            logits    = classifier_lora(embedding)
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels_list.extend(labels.cpu().numpy())

    val_acc = accuracy_score(all_labels_list, all_preds)
    val_f1  = f1_score(all_labels_list, all_preds,
                      average="macro", zero_division=0)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

lora_time = time.time() - lora_start

print(f"\nLoRA RESULTS:")
print(f"Val Accuracy: {val_acc:.4f}")
print(f"Val Macro F1: {val_f1:.4f}")
print(f"Runtime:      {lora_time:.2f}s")
lora_acc, lora_f1 = val_acc, val_f1

STRATEGY 2: LoRA
trainable params: 3,932,160 || all params: 1,212,205,056 || trainable%: 0.3244
  Batch 100/1250 | Loss: 2.6598 | Acc: 36.5%
  Batch 200/1250 | Loss: 2.0865 | Acc: 53.2%
  Batch 300/1250 | Loss: 1.4936 | Acc: 66.9%
  Batch 400/1250 | Loss: 1.1789 | Acc: 73.5%
  Batch 500/1250 | Loss: 0.9718 | Acc: 77.9%
  Batch 600/1250 | Loss: 0.8312 | Acc: 81.0%
  Batch 700/1250 | Loss: 0.7267 | Acc: 83.3%
  Batch 800/1250 | Loss: 0.6501 | Acc: 85.0%
  Batch 900/1250 | Loss: 0.5863 | Acc: 86.4%
  Batch 1000/1250 | Loss: 0.5388 | Acc: 87.5%
  Batch 1100/1250 | Loss: 0.4997 | Acc: 88.4%
  Batch 1200/1250 | Loss: 0.4668 | Acc: 89.1%
Epoch 1/3 | Val Acc: 0.9750 | Val F1: 0.9764
  Batch 100/1250 | Loss: 0.0794 | Acc: 96.8%
  Batch 200/1250 | Loss: 0.0681 | Acc: 97.9%
  Batch 300/1250 | Loss: 0.0622 | Acc: 98.0%
  Batch 400/1250 | Loss: 0.0629 | Acc: 98.0%
  Batch 500/1250 | Loss: 0.0622 | Acc: 98.1%
  Batch 600/1250 | Loss: 0.0598 | Acc: 98.2%
  Batch 700/1250 | Loss: 0.0574 | Acc: 98.2%
 